# Data join exploration

This notebook combines the available 1BM130 data sources into analysis-ready tables. The main output is a neighbourhood-level table with CBS characteristics, neighbourhood/district/municipality names, and postcode counts. It also includes an optional accessibility join using the large neighbourhood-to-neighbourhood travel-time file.

## Merge overview

The main merge level is the Dutch neighbourhood (`buurt`). The source files use slightly different versions of the same geographic identifiers, so the notebook standardises them before joining:

- `buurt_2025.csv` is the base neighbourhood lookup. It contains `Buurt2025`, for example `00140000`.
- `wijk_2025.csv` is joined through the first six digits of `Buurt2025`, converted to `wk_code`, for example `WK001400`. This adds the district (`wijk`) name.
- `gem_2025.csv` is joined through the first four digits of `Buurt2025`, converted to `gm_code`, for example `GM0014`. This adds the municipality (`gemeente`) name.
- `kwb2025.xlsx` from CBS uses `gwb_code_10`, for example `BU00140000`. This is the same neighbourhood ID as `Buurt2025`, but with the `BU` prefix. The notebook renames it to `bu_code` and joins CBS population, housing, income, density, urbanity, and amenity-distance variables.
- `pc6hnr20250801_gwb.csv` contains postcode-house-number records with `Buurt2025`. These are aggregated to counts per neighbourhood, converted to `bu_code`, and joined onto the neighbourhood table.
- `buurt_to_buurt.csv` contains origin-destination travel times. Its `buurt_ori_id` already has the `BU...` format, so it is renamed to `bu_code` and joined to the neighbourhood master table. This step runs on a sample by default because the file is very large.
- `ODiN2024_DANS_Databestand_ Updated.xlsx` is person/trip-level survey data. It does not join directly to neighbourhoods, but it has residential municipality `Wogem_DANS24`. The notebook converts that to `gm_code`, aggregates travel indicators per municipality, and joins those to the municipality lookup.

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 160)

BASE = Path.cwd()
DATA = BASE / 'data'
EXTRA = DATA / 'Extra data'
CBS = DATA / 'CBS'
ODIN = DATA / 'OdiN' / 'ODiN2024 Updated with Header'

print(BASE)
print('Data folder exists:', DATA.exists())

ModuleNotFoundError: No module named 'numpy'

## 1. Load geographic lookup tables

The extra lookup files store geographic codes without prefixes, for example `00140000`. CBS stores the same neighbourhood with a prefix, for example `BU00140000`. This section creates explicit `bu_code`, `wk_code`, and `gm_code` columns so later joins use clear, comparable IDs.

In [ ]:
buurt = pd.read_csv(EXTRA / 'buurt_2025.csv', sep=';', dtype=str)
wijk = pd.read_csv(EXTRA / 'wijk_2025.csv', sep=';', dtype=str)
gemeente = pd.read_csv(EXTRA / 'gem_2025.csv', sep=';', dtype=str)

buurt['bu_code'] = 'BU' + buurt['Buurt2025']
buurt['wk_code'] = 'WK' + buurt['Buurt2025'].str.slice(0, 6)
buurt['gm_code'] = 'GM' + buurt['Buurt2025'].str.slice(0, 4)

wijk['wk_code'] = 'WK' + wijk['Wijk2025']
wijk['gm_code'] = 'GM' + wijk['Wijk2025'].str.slice(0, 4)

gemeente['gm_code'] = 'GM' + gemeente['Gemeente2025']

geo_lookup = (
    buurt
    .merge(wijk[['wk_code', 'wijknaam2025']], on='wk_code', how='left')
    .merge(gemeente[['gm_code', 'Gemeentenaam2025']], on='gm_code', how='left')
)

geo_lookup.head()

## 2. Load CBS neighbourhood characteristics

CBS uses `.` for missing values. This section replaces those placeholders with real missing values and converts the selected numeric columns to numbers. The join key is `bu_code`, created from CBS `gwb_code_10`.

In [ ]:
cbs_cols = [
    'gwb_code_10', 'gwb_code_8', 'regio', 'gm_naam', 'recs',
    'a_inw', 'bev_dich', 'g_ink_pi', 'g_wozbag', 'p_koopw', 'p_huurw',
    'a_woning', 'g_afs_hp', 'g_afs_gs', 'g_afs_kv', 'g_afs_sc', 'g_3km_sc',
    'a_opp_ha', 'ste_mvs'
]

cbs_raw = pd.read_excel(CBS / 'kwb2025.xlsx', usecols=cbs_cols, dtype={'gwb_code_10': str, 'gwb_code_8': str})
cbs_buurt = cbs_raw[cbs_raw['recs'].eq('Buurt')].copy()
cbs_buurt = cbs_buurt.replace('.', pd.NA)

numeric_cols = [c for c in cbs_buurt.columns if c not in ['gwb_code_10', 'gwb_code_8', 'regio', 'gm_naam', 'recs']]
for col in numeric_cols:
    cbs_buurt[col] = pd.to_numeric(cbs_buurt[col], errors='coerce')

cbs_buurt = cbs_buurt.rename(columns={'gwb_code_10': 'bu_code'})
cbs_buurt.head()

## 3. Count PC6 addresses per neighbourhood

The PC6 file is large, so it is processed in chunks. The notebook counts postcode-house-number rows per `Buurt2025`, converts that ID to `bu_code`, and later joins those counts to the neighbourhood master table. `pc6_unique_approx` is an approximate unique postcode count because uniqueness is calculated within chunks before summing.

In [ ]:
pc6_path = EXTRA / 'pc6hnr20250801_gwb.csv'
pc6_parts = []

for chunk in pd.read_csv(pc6_path, sep=';', dtype={'Buurt2025': str, 'PC6': str}, usecols=['PC6', 'Buurt2025'], chunksize=500_000):
    part = (
        chunk
        .dropna(subset=['Buurt2025'])
        .groupby('Buurt2025')
        .agg(pc6_rows=('PC6', 'size'), pc6_unique_approx=('PC6', 'nunique'))
        .reset_index()
    )
    pc6_parts.append(part)

pc6_counts = pd.concat(pc6_parts, ignore_index=True)
pc6_counts = (
    pc6_counts
    .groupby('Buurt2025', as_index=False)
    .agg(pc6_rows=('pc6_rows', 'sum'), pc6_unique_approx=('pc6_unique_approx', 'sum'))
)
pc6_counts['bu_code'] = 'BU' + pc6_counts['Buurt2025']

pc6_counts.head()

## 4. Build the neighbourhood master table

This table is the first concrete output. It starts from the neighbourhood lookup, then joins CBS neighbourhood characteristics on `bu_code`, and finally joins PC6 postcode counts on the same `bu_code`. The result keeps the source names as well as standardised IDs for later analysis.

In [ ]:
buurt_master = (
    geo_lookup
    .merge(cbs_buurt, on='bu_code', how='left', suffixes=('', '_cbs'))
    .merge(pc6_counts[['bu_code', 'pc6_rows', 'pc6_unique_approx']], on='bu_code', how='left')
)

buurt_master['pc6_rows'] = buurt_master['pc6_rows'].fillna(0).astype('int64')
buurt_master['pc6_unique_approx'] = buurt_master['pc6_unique_approx'].fillna(0).astype('int64')

print(buurt_master.shape)
buurt_master.head()

In [ ]:
buurt_master[['bu_code', 'buurtnaam2025', 'wijknaam2025', 'Gemeentenaam2025', 'a_inw', 'bev_dich', 'g_wozbag', 'pc6_rows', 'pc6_unique_approx']].sample(10, random_state=1)

## 5. Optional: aggregate neighbourhood-to-neighbourhood accessibility

`buurt_to_buurt.csv` is about 17GB, so the full run is disabled by default. Set `RUN_FULL_ACCESSIBILITY = True` if you want to aggregate the complete file. With the default setting, only a quick sample is read to test the join structure. The origin ID `buurt_ori_id` is renamed to `bu_code`, which matches the neighbourhood master table.

In [ ]:
RUN_FULL_ACCESSIBILITY = False
btb_path = EXTRA / 'buurt_to_buurt.csv'

usecols = ['buurt_ori_id', 'buurt_id', 'bike_2025_smart_minutes', 'bike_2025_smart_distance']

if RUN_FULL_ACCESSIBILITY:
    parts = []
    for chunk in pd.read_csv(btb_path, usecols=usecols, chunksize=1_000_000):
        chunk = chunk.dropna(subset=['bike_2025_smart_minutes'])
        chunk['bike_within_10min'] = chunk['bike_2025_smart_minutes'].le(10)
        chunk['bike_within_15min'] = chunk['bike_2025_smart_minutes'].le(15)
        chunk['bike_minutes_sum'] = chunk['bike_2025_smart_minutes']
        chunk['bike_distance_sum'] = chunk['bike_2025_smart_distance']
        part = chunk.groupby('buurt_ori_id').agg(
            destinations_observed=('buurt_id', 'size'),
            bike_dest_10min=('bike_within_10min', 'sum'),
            bike_dest_15min=('bike_within_15min', 'sum'),
            bike_minutes_sum=('bike_minutes_sum', 'sum'),
            bike_distance_sum=('bike_distance_sum', 'sum'),
        )
        parts.append(part)

    accessibility = pd.concat(parts).groupby(level=0).sum(numeric_only=True).reset_index()
    accessibility['bike_minutes_mean'] = accessibility['bike_minutes_sum'] / accessibility['destinations_observed']
    accessibility['bike_distance_mean'] = accessibility['bike_distance_sum'] / accessibility['destinations_observed']
else:
    sample = pd.read_csv(btb_path, usecols=usecols, nrows=250_000)
    sample = sample.dropna(subset=['bike_2025_smart_minutes'])
    sample['bike_within_10min'] = sample['bike_2025_smart_minutes'].le(10)
    sample['bike_within_15min'] = sample['bike_2025_smart_minutes'].le(15)
    accessibility = sample.groupby('buurt_ori_id').agg(
        destinations_observed=('buurt_id', 'size'),
        bike_dest_10min=('bike_within_10min', 'sum'),
        bike_dest_15min=('bike_within_15min', 'sum'),
        bike_minutes_mean=('bike_2025_smart_minutes', 'mean'),
        bike_distance_mean=('bike_2025_smart_distance', 'mean'),
    ).reset_index()

accessibility = accessibility.drop(columns=['bike_minutes_sum', 'bike_distance_sum'], errors='ignore')
accessibility = accessibility.rename(columns={'buurt_ori_id': 'bu_code'})
accessibility.head()

In [ ]:
buurt_with_accessibility = buurt_master.merge(accessibility, on='bu_code', how='left')
buurt_with_accessibility[['bu_code', 'buurtnaam2025', 'Gemeentenaam2025', 'a_inw', 'bike_dest_10min', 'bike_dest_15min', 'bike_minutes_mean']].head(10)

## 6. ODiN 2024 municipality-level travel indicators

ODiN is person/trip-level travel survey data and does not directly connect to neighbourhood IDs. It does contain residential municipality `Wogem_DANS24`. The notebook converts that municipality code to `gm_code`, aggregates weighted travel indicators per municipality, and joins the result to the municipality lookup.

In [ ]:
odin_cols = ['Wogem_DANS24', 'Verpl', 'Hvm', 'KHvm', 'AfstV', 'Reisduur', 'FactorV', 'Doel', 'HHGestInkG']
odin = pd.read_excel(ODIN / 'ODiN2024_DANS_Databestand_ Updated.xlsx', usecols=odin_cols)

trips = odin[odin['Verpl'].eq(1)].copy()
trips['wV'] = pd.to_numeric(trips['FactorV'], errors='coerce').fillna(0)
trips['dist_km'] = trips['AfstV'] / 10
trips['is_bike'] = trips['KHvm'].eq(5)
trips['within10min'] = trips['Reisduur'].le(10)
trips['is_essential'] = trips['Doel'].isin([7, 8, 13])
trips['gm_code'] = 'GM' + trips['Wogem_DANS24'].astype('Int64').astype(str).str.zfill(4)

trips['bike_w'] = trips['is_bike'].astype(float) * trips['wV']
trips['local_w'] = trips['within10min'].astype(float) * trips['wV']
trips['essential_w'] = trips['is_essential'].astype(float) * trips['wV']
trips['dist_w'] = trips['dist_km'] * trips['wV']

gemeente_odin = trips.groupby('gm_code', as_index=False).agg(
    weighted_trips=('wV', 'sum'),
    bike_w=('bike_w', 'sum'),
    local_w=('local_w', 'sum'),
    essential_w=('essential_w', 'sum'),
    dist_w=('dist_w', 'sum'),
)
gemeente_odin['bike_share_pct'] = gemeente_odin['bike_w'] / gemeente_odin['weighted_trips'] * 100
gemeente_odin['local_trip_share_pct'] = gemeente_odin['local_w'] / gemeente_odin['weighted_trips'] * 100
gemeente_odin['essential_trip_share_pct'] = gemeente_odin['essential_w'] / gemeente_odin['weighted_trips'] * 100
gemeente_odin['avg_dist_km_weighted'] = gemeente_odin['dist_w'] / gemeente_odin['weighted_trips']
gemeente_odin = gemeente_odin.drop(columns=['bike_w', 'local_w', 'essential_w', 'dist_w'])

gemeente_odin = gemeente[['gm_code', 'Gemeentenaam2025']].merge(gemeente_odin, on='gm_code', how='left')
gemeente_odin.sort_values('bike_share_pct', ascending=False).head(15)

## 7. Save merged outputs

The CSV files below can be used as starting points for further analysis or visualisation.

In [ ]:
output_dir = BASE / 'outputs'
output_dir.mkdir(exist_ok=True)

buurt_master.to_csv(output_dir / 'buurt_master.csv', index=False)
buurt_with_accessibility.to_csv(output_dir / 'buurt_with_accessibility_sample.csv', index=False)
gemeente_odin.to_csv(output_dir / 'gemeente_odin_2024.csv', index=False)

print('Saved:')
print(output_dir / 'buurt_master.csv')
print(output_dir / 'buurt_with_accessibility_sample.csv')
print(output_dir / 'gemeente_odin_2024.csv')